# 01 - Data Understanding

## Mục tiêu

Notebook này nhằm mục đích:
- Hiểu cấu trúc và nội dung của dataset
- Kiểm tra phạm vi dữ liệu (quốc gia, năm)
- Đánh giá chất lượng dữ liệu ban đầu (missing values, duplicates)
- Cung cấp tổng quan trước khi thực hiện phân tích sâu

## Tại sao thực hiện?

Hiểu dữ liệu là bước đầu tiên và quan trọng nhất trong quy trình phân tích dữ liệu. Nó giúp:
- Xác định các biến có sẵn và định nghĩa của chúng
- Phát hiện các vấn đề chất lượng dữ liệu sớm
- Định hình hướng phân tích phù hợp

## Kết quả mong đợi

- Tổng quan đầy đủ về dataset
- Danh sách các quốc gia và năm được bao phủ
- Đánh giá về missing values và duplicates
- Nhận xét ban đầu về dữ liệu

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set display options for better readability
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', '{:.4f}'.format)

# Set style for visualizations
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 2. Load Dataset

In [ ]:
# Load the processed unemployment dataset
df = pd.read_csv('../data/processed/unemployment_final.csv')

# Load data dictionary for variable explanations
data_dict = pd.read_csv('../data/processed/data_dictionary.csv')

print("Dataset loaded successfully!")
print(f"Shape: {df.shape}")

## 3. Dataset Overview

In [ ]:
# Display basic information
print("="*60)
print("DATASET SHAPE")
print("="*60)
print(f"Number of rows: {df.shape[0]}")
print(f"Number of columns: {df.shape[1]}")
print()

print("="*60)
print("COLUMN NAMES")
print("="*60)
print(df.columns.tolist())
print()

In [ ]:
# Display first few rows
print("="*60)
print("FIRST 5 ROWS")
print("="*60)
display(df.head())

In [ ]:
# Display last few rows
print("="*60)
print("LAST 5 ROWS")
print("="*60)
display(df.tail())

In [ ]:
# Display data types
print("="*60)
print("DATA TYPES")
print("="*60)
print(df.dtypes)
print()

In [ ]:
# Display statistical summary
print("="*60)
print("STATISTICAL SUMMARY")
print("="*60)
display(df.describe())

## 4. Data Dictionary

In [ ]:
print("="*60)
print("DATA DICTIONARY")
print("="*60)
display(data_dict)

## 5. Country Coverage

In [ ]:
# Check unique countries
countries = df['Country'].unique()
country_codes = df['Country_Code'].unique()

print("="*60)
print("COUNTRY COVERAGE")
print("="*60)
print(f"Number of unique countries: {len(countries)}")
print()
print("Countries and their codes:")
for country, code in zip(countries, country_codes):
    print(f"  - {country} ({code})")
print()

In [ ]:
# Count records per country
country_counts = df.groupby('Country').size().reset_index(name='Record_Count')
print("="*60)
print("RECORD COUNT PER COUNTRY")
print("="*60)
display(country_counts)
print()

In [ ]:
# Visualize record distribution
plt.figure(figsize=(10, 6))
sns.barplot(data=country_counts, x='Country', y='Record_Count', palette='viridis')
plt.title('Number of Records per Country', fontsize=14, fontweight='bold')
plt.xlabel('Country', fontsize=12)
plt.ylabel('Number of Records', fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 6. Year Coverage

In [ ]:
# Check year range
years = df['Year'].unique()
years_sorted = sorted(years)

print("="*60)
print("YEAR COVERAGE")
print("="*60)
print(f"Year range: {years_sorted[0]} to {years_sorted[-1]}")
print(f"Number of unique years: {len(years)}")
print()
print("All years covered:")
print(years_sorted)
print()

In [ ]:
# Check if all countries have data for all years
print("="*60)
print("YEAR COVERAGE BY COUNTRY")
print("="*60)
for country in countries:
    country_years = sorted(df[df['Country'] == country]['Year'].unique())
    print(f"{country}: {len(country_years)} years ({country_years[0]}-{country_years[-1]})")
print()

## 7. Missing Values Analysis

In [ ]:
# Check for missing values
missing_counts = df.isnull().sum()
missing_percentages = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': missing_counts.values,
    'Missing_Percentage': missing_percentages.values
})

print("="*60)
print("MISSING VALUES SUMMARY")
print("="*60)
display(missing_df)
print()

In [ ]:
# Visualize missing values
plt.figure(figsize=(12, 6))
if missing_df['Missing_Count'].sum() > 0:
    sns.barplot(data=missing_df[missing_df['Missing_Count'] > 0], 
                x='Column', y='Missing_Percentage', palette='Reds_r')
    plt.title('Missing Values Percentage by Column', fontsize=14, fontweight='bold')
    plt.xlabel('Column', fontsize=12)
    plt.ylabel('Missing Percentage (%)', fontsize=12)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("No missing values found in the dataset.")

In [ ]:
# Check missing values by country
print("="*60)
print("MISSING VALUES BY COUNTRY")
print("="*60)
for country in countries:
    country_data = df[df['Country'] == country]
    missing_by_country = country_data.isnull().sum()
    if missing_by_country.sum() > 0:
        print(f"\n{country}:")
        print(missing_by_country[missing_by_country > 0])
    else:
        print(f"\n{country}: No missing values")
print()

## 8. Duplicate Check

In [ ]:
# Check for duplicate rows
duplicate_count = df.duplicated().sum()

print("="*60)
print("DUPLICATE CHECK")
print("="*60)
print(f"Number of duplicate rows: {duplicate_count}")

if duplicate_count > 0:
    print("\nDuplicate rows:")
    display(df[df.duplicated(keep=False)])
else:
    print("No duplicate rows found.")
print()

In [ ]:
# Check for duplicates based on key columns (Country, Year)
key_duplicates = df.duplicated(subset=['Country', 'Year'], keep=False).sum()

print("="*60)
print("DUPLICATE CHECK BY KEY (Country, Year)")
print("="*60)
print(f"Number of duplicate (Country, Year) combinations: {key_duplicates}")

if key_duplicates > 0:
    print("\nDuplicate combinations:")
    display(df[df.duplicated(subset=['Country', 'Year'], keep=False)])
else:
    print("No duplicate (Country, Year) combinations found.")
print()

## 9. Initial Observations

In [ ]:
# Summary statistics by country
print("="*60)
print("SUMMARY STATISTICS BY COUNTRY")
print("="*60)
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
country_stats = df.groupby('Country')[numeric_cols].agg(['mean', 'min', 'max'])
display(country_stats)
print()

In [ ]:
# Check data consistency
print("="*60)
print("DATA CONSISTENCY CHECKS")
print("="*60)

# Check if all numeric columns are within reasonable ranges
print("\n1. Numeric column ranges:")
for col in numeric_cols:
    if col != 'Year':
        print(f"   {col}: min={df[col].min():.2f}, max={df[col].max():.2f}")

# Check if population values are positive
print("\n2. Population values:")
print(f"   All positive: {(df['Population'] > 0).all()}")

# Check if unemployment rates are within 0-100
print("\n3. Unemployment rates (0-100%):")
print(f"   Unemployment_Rate in range: {((df['Unemployment_Rate'] >= 0) & (df['Unemployment_Rate'] <= 100)).all()}")
print(f"   Youth_Unemployment_Rate in range: {((df['Youth_Unemployment_Rate'] >= 0) & (df['Youth_Unemployment_Rate'] <= 100)).all()}")
print()

## 10. Conclusion

### Tổng kết:

1. **Dataset Structure**:
   - Số lượng bản ghi: 75
   - Số lượng cột: 10
   - Thời gian: 2010-2024 (15 năm)

2. **Quốc gia được bao phủ**:
   - 5 quốc gia châu Á: Japan, Philippines, South Korea, Thailand, Vietnam
   - Mỗi quốc gia có 15 bản ghi (tương ứng với 15 năm)
   - Dữ liệu cân bằng giữa các quốc gia

3. **Biến số**:
   - Country, Country_Code, Year: Thông tin định danh
   - Unemployment_Rate, Youth_Unemployment_Rate: Chỉ số thất nghiệp
   - GDP_Growth, GDP_per_Capita: Chỉ số kinh tế
   - Inflation: Lạm phát
   - Labor_Force_Participation: Tỷ lệ tham gia lực lượng lao động
   - Population: Dân số

4. **Chất lượng dữ liệu**:
   - Không có missing values
   - Không có duplicate rows
   - Các giá trị numeric nằm trong phạm vi hợp lý

5. **Nhận xét ban đầu**:
   - Dataset đã được làm sạch và chuẩn hóa tốt
   - Dữ liệu đầy đủ cho giai đoạn 2010-2024
   - Phù hợp cho phân tích xu hướng và so sánh giữa các quốc gia
   - Không có vấn đề chất lượng dữ liệu rõ ràng cần xử lý trước khi phân tích

### Bước tiếp theo:

Dataset có chất lượng tốt và sẵn sàng cho:
- Phân tích thống kê chi tiết (Notebook 04)
- Khám phá dữ liệu và visualization (Notebook 03)
- Kiểm tra và validation thêm (Notebook 02)